#### Enviroment Setup

In [1]:
%pip install -r requirements.txt
!playwright install

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


#### Loading dataset and filtering it

In [2]:
import pandas as pd
from bertopic import BERTopic

In [3]:
#Reading csv, extracting useful columns, appending future columns
dataset_path = 'dataset/datasetA.csv'
df = pd.read_csv(dataset_path, na_filter=False)


# Code below may or may not be commented out depending on dataset_path
unnecessary_columns = ['date', 'number_of_characters_title', 'number_of_words_title', 'day_of_week', 'month', 'year', 'quarter', 'is_weekend']
df = df.drop(columns=unnecessary_columns)
df[['text', 'authors', 'keywords', 'summary']] = ''

In [ ]:
#Selecting just the title and classes_str columns. Put in a new dataframe so its less confusing for the rest of the code
titles_and_classes = df[['title', 'classes_str']]
#Checking nulls 
titles_and_classes.isna().sum()

In [ ]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = titles_and_classes[titles_and_classes['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]

#### Testing Various Topic Modeling Techniques

In [ ]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

In [ ]:
#Default BERTopic Model 
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()
"""
Topic is the topic number. -1 refers to outliers so it can be ignored.
Count is how many rows from title fit that topic
Name is the topic name 
Representation is the top words that summarize the topic
Representative_Docs is the example documents that best illustrate the topic"""

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topics = topic_model.reduce_topics(docs, nr_topics=10)  
topic_model.get_topic_info()

### Webscraping

In [38]:
from newspaper import Article
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
# import nest_asyncio
# nest_asyncio.apply()
import time
import asyncio
from playwright.async_api import async_playwright, expect
from playwright.sync_api import sync_playwright
import joblib
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
print(dir(async_playwright))

['__annotate__', '__annotations__', '__builtins__', '__call__', '__class__', '__closure__', '__code__', '__defaults__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__get__', '__getattribute__', '__getstate__', '__globals__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__kwdefaults__', '__le__', '__lt__', '__module__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__type_params__']


In [6]:
async def do_something():
    # p = await async_playwright().start()
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(df['link'][69])
        log.info(await page.url())
        await browser.close()
    # await p.stop()

In [ ]:
# log file for Emmanuel's debuging ventures
import logging
logging.basicConfig(filename='scraperLog.log', format='%(levelname)s: %(asctime)s: %(message)s', datefmt='%m/%d/%Y - %I:%M:%S %p', level=logging.DEBUG)
logging.getLogger('selenium').setLevel(logging.WARNING)
log = logging.getLogger(__name__)
#levels: debug, info, warning, error, critical

In [60]:
# all selenium webdriver setting for max efficiency
def setup_driver():
    driver = None
    
    try: 
        chrome_options = Options()
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--disable-renderer-backgrounding")
        chrome_options.add_argument("--disable-background-timer-throttling")
        chrome_options.add_argument("--disable-backgrounding-occluded-windows")
        chrome_options.add_argument("--disable-dev-shm-usage")
        
        # service = Service(timeout=15)
        driver = webdriver.Chrome(chrome_options)
        
        # driver.set_page_load_timeout = 15
    except Exception as e:
        log.info('Webdriver setup failed.')
        log.error(e)
   
    return driver

#for cases of HTTPConnectionPool timeout
def reset_driver(driver, retries = 2):
    try:
        driver.quit()
        newDriver = setup_driver()
        log.info("Driver reset.")
        return newDriver
    except Exception as e:
        log.info("Failed to reset driver.")
        log.info(e)
        if retries > 0:
            log.info('Retrying')
            reset_driver(driver, retries - 1)


In [ ]:
#selenium is garbage and this roundabout is only way i've found to apply WebDriverWait effectively with time.sleep()
class wait_for(object):

    def __init__(self, later):
        self.later = later

    def __call__(self, driver):
        if time.time() < self.later:
            return False
        return True

In [ ]:
# parses the javascript of the html to redirect and retrieve the correct url
def retrieve_url(driver, link: str, row_index: int = -1, wait_time: int = 0.5) -> (str | None):
    url = None

    try:
        driver.get(link)
        later = time.time() + 0.4
        WebDriverWait(driver, 30, wait_time).until(wait_for(later))

        url = driver.current_url

        if url.startswith('https://news.google.com'):   
            raise Exception("Redirect not completed.")
    except Exception as e:
            if wait_time < 1:
                log.info(f'Retrying url for at row {row_index}.')
                retrieve_url(driver, link, row_index, 3)
            else:
                log.info(f"Couldn't retrieve url at row {row_index}.") if row_index > -1 else log.info("Couldn't retrieve url.")
                log.error(e)


    return url

In [ ]:
# return dictionary of authors, keywords, summary, and text
def scrape(url: str, row_index: int = -1) -> (dict[str | list[str]] | None):
    try:
        article = Article(url)
        article.build()
        text = article.text

        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')
        
        return {'authors': article.authors,
                'keywords': article.keywords,
                'summary': article.summary,
                'text': text}
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return

#returns only the text
def scrape_text(url: str, row_index: int = -1) -> (str):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')
        return text
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return "Failed to scrape"
    

In [ ]:
#chrome options setup for headless browser and high efficiency
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-renderer-backgrounding")
chrome_options.add_argument("--disable-background-timer-throttling")
chrome_options.add_argument("--disable-backgrounding-occluded-windows")
chrome_options.add_argument("--disable-dev-shm-usage")

In [ ]:
# input the information and class that you want to scrape, will create a new updated csv
def create_scraped_dataset(required_class: str, filepath: str, NUM_ROWS = 10500) -> None:
    with webdriver.Chrome(chrome_options) as driver:
        for row in range(NUM_ROWS):
            ENTRY_IS_BLANK = (not bool(df['text'][row]))

            if required_class.lower() in df['classes_str'][row].lower() and ENTRY_IS_BLANK: 
                link = df['link'][row]
                url = retrieve_url(driver, link, row)

                #reset driver
                if url == None:
                    log.info(f'Sleeping and resetting driver at row {row}.')
                    driver = reset_driver(driver)
                    log.info(f'Second retrieve_url attempt at row {row}.')
                    url = retrieve_url(driver, link, row)
                    
                #url still equal to 'None' after second attempt - pack it up, ggs
                if url == None:
                    data = "Selenium error - can't access url"
                else:
                    data = scrape_text(url, row)

                df['text'][row] = data
            
            log.debug(f'Row {row} completed.')

            #Process in batches of 100
            if (row + 1) % 100 == 0:
                log.info(f'{row + 1} rows completed.')
                df.to_csv(filepath, index=False)
                # driver = reset_driver(driver)
                # log.info(f'Driver reset; output file updated; batch completed.')

        df.to_csv(filepath, index=False)
        log.info("Full table walkthrough completed.")

In [ ]:
CLASS_TO_SCRAPE = 'Learning, Knowledge & Education'
OUTPUT_FILEPATH = 'dataset/newUpdatedDatasetA.csv'
create_scraped_dataset(CLASS_TO_SCRAPE, OUTPUT_FILEPATH)